In [7]:
import arcpy
import pandas as pd
import numpy as np

# 1. Enable Spatial Analyst extension
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# 2. Define inputs
points_shp = r"N:\Research\Kampf\Private\KeenanW\CO_natural_streamflow\data\terrain\dem_ucol_big50\snapped.shp"
streams_tif = r"N:\Research\Kampf\Private\KeenanW\CO_natural_streamflow\data\terrain\dem_ucol_big50\streams.tif"
gage_field = "gage"
output_csv = r"N:\Research\Kampf\Private\KeenanW\CO_natural_streamflow\data\terrain\dem_ucol_big50\stream_distance_matrix.csv"

# 3. Read gage IDs from point feature class
gages = []
with arcpy.da.SearchCursor(points_shp, [gage_field]) as cursor:
    for row in cursor:
        gages.append(str(row[0]))

print(f"Loaded {len(gages)} gages: {gages}")

# Initialize distance matrix DataFrame
dist_df = pd.DataFrame(index=gages, columns=gages, dtype=float)
gages = gages[0:10]
field_delim = arcpy.AddFieldDelimiters(points_shp, gage_field)


Loaded 285 gages: ['09010500', '09014050', '09015000', '09019000', '09019500', '09021000', '09022000', '09024000', '09025000', '09025300', '09026500', '09027100', '09032000', '09032050', '09032100', '09032200', '09032300', '09032400', '09032990', '09033100', '09033300', '09034250', '09034900', '09035500', '09035700', '09035900', '09036000', '09037500', '09038500', '09040500', '09041090', '09041400', '09046490', '09046600', '09047500', '09047700', '09050100', '09050700', '09051050', '09053500', '09057500', '09058000', '09058500', '09059500', '09060770', '09060799', '09061600', '09063000', '09063900', '09064000', '09064600', '09065100', '09065500', '09066000', '09066200', '09066300', '09066325', '09066510', '09067000', '09067020', '09067200', '09070000', '09070500', '09071295', '09072550', '09073005', '09073300', '09073400', '09074500', '09076000', '09076300', '09078141', '09078475', '09079450', '09080400', '09081000', '09081600', '09085000', '09085100', '09085500', '09087500', '09090785

In [10]:
# 4. Loop through each gage point as a source
for gage_id in gages:
    print(f"Processing source gage: {gage_id}...")
    
    # Select single point as source
    where_clause = f"{arcpy.AddFieldDelimiters(points_shp, gage_field)} = '{gage_id}'"
    # Handling numeric gage values if needed:
    try:
        # Check if integer gage ID
        int(gage_id)
        where_clause = f"{arcpy.AddFieldDelimiters(points_shp, gage_field)} = '{gage_id}'"
    except ValueError:
        pass

    src_point = f"src_{gage_id}"
    arcpy.analysis.Select(points_shp, src_point, where_clause)
    
    # Compute Distance Accumulation along stream network
    out_accum = arcpy.sa.DistanceAccumulation(
        in_source_data=src_point,
        in_cost_raster=streams_tif
    )
    
    # Extract accumulated stream distances at all gage locations
    sample_table = f"sample_{gage_id}"
    arcpy.sa.Sample(
        in_rasters=out_accum,
        in_location_data=points_shp,
        out_table=sample_table,
        resampling_type="NEAREST"
    )
    
    # Read sampled distances into matrix row
    # Sample tool outputs field matching raster name or band name (typically Band_1 or raster name)
    sample_fields = [f.name for f in arcpy.ListFields(sample_table)]
    val_field = [f for f in sample_fields if f not in [gage_field, 'OBJECTID', 'RID', 'X', 'Y']][0]
    
    with arcpy.da.SearchCursor(sample_table, [gage_field, val_field]) as cursor:
        for row in cursor:
            target_gage = str(row[0])
            dist_val = row[1]
            
            # Distance Accumulation returns NoData / negative values for unconnected stream segments
            if dist_val is None or dist_val < 0:
                dist_df.loc[gage_id, target_gage] = np.nan
            else:
                dist_df.loc[gage_id, target_gage] = round(float(dist_val), 2)
    
    # Clean up temporary in-memory layers
    arcpy.management.Delete(src_point)
    arcpy.management.Delete(out_accum)
    arcpy.management.Delete(sample_table)

# 5. Save final matrix to CSV
dist_df.to_csv(output_csv)
print(f"\nDistance matrix successfully created and saved to:\n{output_csv}")

# Return Spatial Analyst license
arcpy.CheckInExtension("Spatial")

Processing source gage: 09010500...


<class 'RuntimeError'>: Cannot find field 'gage'

In [11]:
sample_table

'sample_09010500'